In [0]:
secrets_path='/Volumes/databricks_learning/custom/secrets'
bootstrap_servers ='kafka-instance-bipin-data-engineering-project-1.j.aivencloud.com:10670'
topic='power-meter-simulator'
iot_raw_table_name='databricks_learning.custom.iot_raw_measurements'
# secrets_path=''
# bootstrap_servers =''
# topic=''
# iot_raw_table_name=''

dbutils.widgets.text(name='bootstrap_servers',defaultValue=bootstrap_servers, label='bootstrap_servers')
dbutils.widgets.text(name='topic',defaultValue=topic, label='topic')
dbutils.widgets.text(name='iot_raw_table_name',defaultValue=iot_raw_table_name, label='iot_raw_table_name')
dbutils.widgets.text(name='secrets_path',defaultValue=secrets_path, label='secrets_path')

bootstrap_servers = dbutils.widgets.get('bootstrap_servers')
topic = dbutils.widgets.get('topic')
bronze_table_name = dbutils.widgets.get('iot_raw_table_name')
secrets_path = dbutils.widgets.get('secrets_path')

In [0]:
import random
from datetime import datetime, timedelta, timezone

def generate_meter_reading(ts, energy_kwh):
    voltage_r = round(random.uniform(228, 234), 2)
    voltage_y = round(random.uniform(228, 234), 2)
    voltage_b = round(random.uniform(228, 234), 2)

    current_r = round(random.uniform(40, 50), 2)
    current_y = round(random.uniform(40, 50), 2)
    current_b = round(random.uniform(40, 50), 2)

    active_power_kw = round(random.uniform(28, 35), 2)
    reactive_power_kvar = round(random.uniform(4, 7), 2)

    apparent_power_kva = round(
        (active_power_kw**2 + reactive_power_kvar**2) ** 0.5,
        2
    )

    power_factor = round(
        active_power_kw / apparent_power_kva,
        3
    )

    energy_kwh += round(active_power_kw / 60, 3)

    return {
        "measurement_time": ts.isoformat(),
        "voltage_r": voltage_r,
        "voltage_y": voltage_y,
        "voltage_b": voltage_b,
        "current_r": current_r,
        "current_y": current_y,
        "current_b": current_b,
        "active_power_kw": active_power_kw,
        "reactive_power_kvar": reactive_power_kvar,
        "apparent_power_kva": apparent_power_kva,
        "power_factor": power_factor,
        "frequency_hz": round(random.uniform(49.8, 50.2), 2),
        "energy_kwh": energy_kwh,
        "thd_voltage": round(random.uniform(1.0, 3.0), 2),
        "thd_current": round(random.uniform(2.0, 5.0), 2),
        "temperature_c": round(random.uniform(35, 42), 2),
    }, energy_kwh

In [0]:
device_id = "METER_001"

end_time = datetime.now(timezone.utc)
start_time = end_time - timedelta(minutes=14)

energy_kwh = 10562.4
readings = []

current_time = start_time

for _ in range(15):
    reading, energy_kwh = generate_meter_reading(
        current_time,
        energy_kwh
    )

    readings.append(reading)

    current_time += timedelta(minutes=1)

event = {
    "device_id": device_id,
    "generated_at": end_time.isoformat(),
    "readings": readings
}

In [0]:
%sh
pip install --upgrade kafka-python
dbutils.library.restartPython()

In [0]:
from kafka import KafkaProducer
import json

producer = KafkaProducer(
    bootstrap_servers=bootstrap_servers,
    security_protocol="SSL",
    ssl_cafile=f"{secrets_path}/ca_certificate.txt",
    ssl_certfile=f"{secrets_path}/access_certificate.txt",
    ssl_keyfile=f"{secrets_path}/access_key.txt",
    key_serializer=lambda k: k.encode("utf-8"),
    value_serializer=lambda v: json.dumps(v).encode("utf-8")
)

producer.send(
    topic=topic,
    key=device_id,
    value=event
)

producer.flush()
producer.close()